In [ ]:
from google.colab import files
import zipfile
import os

# Upload ZIP file
uploaded = files.upload()

# Get uploaded ZIP filename
zip_file = next(iter(uploaded))

# Extract ZIP
extract_folder = "/content/ev_battery_dataset"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("✅ File uploaded and extracted successfully!")
print("\nFiles:")

for file in os.listdir(extract_folder):
    print("-", file)

Saving EV Battery Failure Prediction Dataset.zip to EV Battery Failure Prediction Dataset.zip
✅ File uploaded and extracted successfully!

Files:
- ev_battery_failure_dataset.csv
- data_dictionary.csv
- feature_descriptions.md


In [ ]:
import pandas as pd

file_path = "/content/ev_battery_dataset/ev_battery_failure_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (200000, 70)

Columns:
['vehicle_id', 'vehicle_brand', 'vehicle_model', 'vehicle_type', 'manufacturing_year', 'battery_manufacturer', 'battery_chemistry', 'battery_capacity_kwh', 'drive_type', 'odometer_km', 'vehicle_age_years', 'fleet_or_private', 'battery_serial', 'cycle_count', 'battery_health_percent', 'state_of_charge', 'depth_of_discharge', 'state_of_health', 'cell_voltage_avg', 'cell_voltage_std', 'pack_voltage', 'cell_temperature_avg', 'cell_temperature_max', 'internal_resistance', 'charge_efficiency', 'discharge_efficiency', 'remaining_capacity', 'capacity_loss_percent', 'charging_cycles_last_month', 'fast_charge_ratio', 'slow_charge_ratio', 'average_charge_power_kw', 'average_charging_time', 'overnight_charging_ratio', 'home_charging_ratio', 'charging_interruptions', 'overcharge_events', 'average_speed', 'average_trip_distance', 'aggressive_acceleration_score', 'hard_braking_score', 'regenerative_braking_usage', 'highway_driving_ratio', 'city_driving_ratio', 'd

,vehicle_id,vehicle_brand,vehicle_model,vehicle_type,manufacturing_year,battery_manufacturer,battery_chemistry,battery_capacity_kwh,drive_type,odometer_km,...,sensor_fault_count,BMS_warning_count,abnormal_voltage_events,battery_stress_index,aging_score,thermal_health_score,charging_quality_score,driving_stress_score,predicted_remaining_life_cycles,battery_failure
0,EV100000,Nissan,Leaf,SUV,2019.0,CATL,LMO,82.54,RWD,98163.0,...,2.0,3.0,0.0,0.0,53.5,66.0,65.2,0.0,401.0,0
1,EV100001,Audi,e-tron,Sedan,2018.0,Samsung SDI,NaN,64.17,AWD,113600.0,...,0.0,3.0,0.0,51.0,38.8,67.8,46.3,40.9,1094.0,0
2,EV100002,Toyota,bZ4X,Van,2018.0,Guoxuan,NMC,90.92,RWD,242253.0,...,4.0,1.0,0.0,36.2,66.1,48.6,0.0,6.5,645.0,0
3,EV100003,Volkswagen,ID.3,Hatchback,2022.0,CATL,NMC,47.35,AWD,83700.0,...,0.0,3.0,1.0,23.4,32.5,58.2,NaN,NaN,867.0,0
4,EV100004,Tesla,Model X,Crossover,2015.0,BYD Battery,NMC,68.04,FWD,88784.0,...,3.0,2.0,1.0,51.2,45.4,51.0,NaN,100.0,1090.0,0


In [ ]:
print(df.isnull().sum())

vehicle_id                            0
vehicle_brand                      6436
vehicle_model                      8204
vehicle_type                       6432
manufacturing_year                 9829
                                   ... 
thermal_health_score               6329
charging_quality_score             7186
driving_stress_score               6810
predicted_remaining_life_cycles    8974
battery_failure                       0
Length: 70, dtype: int64


In [ ]:
print(df["battery_failure"].value_counts())

battery_failure
0    180079
1     19921
Name: count, dtype: int64


In [ ]:
X = df.drop("battery_failure", axis=1)
y = df["battery_failure"]

print(X.shape)
print(y.shape)

(200000, 69)
(200000,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(160000, 69)
(40000, 69)


In [ ]:
# Remove ID column
X_train = X_train.drop("vehicle_id", axis=1, errors="ignore")
X_test = X_test.drop("vehicle_id", axis=1, errors="ignore")

# Convert categorical columns
from sklearn.preprocessing import LabelEncoder

for col in X_train.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

print("Preprocessing completed!")
print(X_train.shape)

Preprocessing completed!
(160000, 68)


In [ ]:
print(X_train.select_dtypes(include="object").columns.tolist())

[]


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Remove ID columns
id_cols = ["vehicle_id", "battery_id"]

X_train = X_train.drop(columns=id_cols, errors="ignore")
X_test = X_test.drop(columns=id_cols, errors="ignore")

# Encode remaining categorical columns
for col in X_train.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Preprocessing completed!")

X_train: (160000, 68)
X_test: (40000, 68)
Preprocessing completed!


In [ ]:
print(X_train.select_dtypes(include="object").columns)
print(X_test.select_dtypes(include="object").columns)

Index([], dtype='object')
Index(['battery_serial'], dtype='object')


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training started...")

model.fit(X_train, y_train)

print("Training completed!")
print("Trees:", len(model.estimators_))

Training started...
Training completed!
Trees: 100


In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Accuracy %:", accuracy_score(y_test, y_pred) * 100)

In [32]:
from sklearn.preprocessing import LabelEncoder

# Find text columns
cat_cols = X_train.select_dtypes(include="object").columns

print("Text columns:", cat_cols.tolist())

# Encode
for col in cat_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))

    X_test[col] = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

print("Encoding completed!")
print("Remaining text columns:")
print(X_test.select_dtypes(include="object").columns.tolist())

Text columns: ['terrain_type']
Encoding completed!
Remaining text columns:
['drive_type', 'battery_serial']


In [38]:
import joblib

joblib.dump(fast_model, "/content/ev_battery_failure_model.pkl")

print("Model saved!")

Model saved!


In [39]:
import joblib

model = joblib.load("/content/ev_battery_failure_model.pkl")

sample = X_test.iloc[[0]]

prediction = model.predict(sample)[0]

print("Battery Failure Prediction:", prediction)
print("Result:", "FAILURE" if prediction == 1 else "NO FAILURE")

Battery Failure Prediction: 0
Result: NO FAILURE


In [41]:
samples = X_test.iloc[:5]

predictions = model.predict(samples)

for i, p in enumerate(predictions, 1):
    print(f"Battery {i}: {'FAILURE' if p == 1 else 'NO FAILURE'}")

Battery 1: NO FAILURE
Battery 2: NO FAILURE
Battery 3: NO FAILURE
Battery 4: NO FAILURE
Battery 5: NO FAILURE


In [42]:
# Take one existing battery from test data

sample = X_test.iloc[[0]]

prediction = model.predict(sample)[0]

print("Battery Failure Prediction:")

if prediction == 1:
    print("FAILURE")
else:
    print("NO FAILURE")

Battery Failure Prediction:
NO FAILURE


In [43]:
samples = X_test.iloc[:5]

predictions = model.predict(samples)

for i, p in enumerate(predictions, 1):
    print(f"Battery {i}: {'FAILURE' if p == 1 else 'NO FAILURE'}")

Battery 1: NO FAILURE
Battery 2: NO FAILURE
Battery 3: NO FAILURE
Battery 4: NO FAILURE
Battery 5: NO FAILURE


In [44]:
import pandas as pd

result = X_test.iloc[:5].copy()
result["Prediction"] = model.predict(X_test.iloc[:5])

print(result[["Prediction"]])

        Prediction
179921           0
69410            0
31083            0
175661           0
36449            0


In [45]:
result.to_csv("battery_predictions.csv", index=False)

print("Predictions saved successfully!")

Predictions saved successfully!


In [46]:
from google.colab import files

files.download("battery_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
print("Project completed successfully!")
print("Prediction file: battery_predictions.csv")
print("Model file: ev_battery_failure_model.pkl")

Project completed successfully!
Prediction file: battery_predictions.csv
Model file: ev_battery_failure_model.pkl
